In [ ]:
import pybliometrics
from pybliometrics.scopus import  ScopusSearch
from pybliometrics.scopus import AffiliationSearch
import pandas as pd
pybliometrics.init()
pybliometrics.scopus.init()

In [ ]:
from pybliometrics.scopus import AbstractRetrieval

search = ScopusSearch('AFFIL("West Virginia University") AND TITLE-ABS-KEY("machine learning" OR "artificial intelligence") AND PUBYEAR > 2018', refresh=True)
import time
import pandas as pd

eids = search.get_eids()  # list of EIDs from the search



rows = []

for eid in eids:
    try:
        ab = AbstractRetrieval(eid, view="FULL")

        funding = getattr(ab, "funding", [])
        funding_text = getattr(ab, "funding_text", "")
        grant_numbers = getattr(ab, "grant_numbers", [])
        funding_ids = getattr(ab, "funding_ids", [])

        rows.append({
            "eid": eid,
            "title": ab.title,
            "funding": "; ".join(funding) if funding else "",
            "funding_text": funding_text,
            "grant_numbers": "; ".join(grant_numbers) if grant_numbers else "",
            "funding_ids": "; ".join(funding_ids) if funding_ids else "",
        })

    except Exception as e:
        print(f"❌ Failed on {eid}: {e}")

    time.sleep(0.2)


funding_df = pd.DataFrame(rows)
funding_df.to_csv("wvu_ml_funding_scopus.csv", index=False)
print("Saved funding data to wvu_ml_funding_scopus.csv")


In [ ]:
from pybliometrics.scopus import AffiliationSearch

query1 = "AFFIL(West Virginia University)"
s = AffiliationSearch(query1)
s.affiliations
df1 = pd.DataFrame(s.affiliations)
df1[df1['city'] == 'Morgantown']

In [ ]:
wvu_faculty = pd.read_excel("/home/kehinde/Elsevier/Faculty_TT_Ten_July_2025.xlsx")
wvu_faculty["Full Name"]

Papers with ML, Deep Learning, AI, or NN in abstract

In [ ]:
query = (
    'AFFIL("West Virginia University") AND ''TITLE-ABS-KEY("Machine Learning" OR "deep learning" OR "artificial intelligence" OR "neural network")'
)
search = ScopusSearch(query, view="COMPLETE")

print("Number of results:", search.get_results_size())
df = pd.DataFrame(search.results)

filtered_df = df
print(filtered_df[['authkeywords','fund_no','fund_acr', 'description','author_names']].nunique())


filtered_df

In [ ]:
OUTFILE = "/home/kehinde/Elsevier/WVU_ML_AI_DL_DeepLearning_Scopus_2025.csv"
filtered_df.to_csv(OUTFILE, index=False)

In [ ]:
filtered_df.columns

In [ ]:
filtered_df[['authkeywords','fund_no','fund_acr', 'description','author_names']].head(30)

In [ ]:
filtered_df[filtered_df["author_names"].str.contains("Aldo", na=False)]

In [ ]:
filtered_df["author_names"]

In [ ]:
# Split by semicolon and strip whitespace
filtered_df["author_names"] = (
    filtered_df["author_names"]
    .astype(str)
    .str.split(";")
    .apply(lambda lst: [a.strip() for a in lst if a.strip()])
)

filtered_df["author_names"]

In [ ]:
# filtered_df.loc[:, "author_names"] = (
#     filtered_df["author_names"]
#         .fillna("")
#         .astype(str)
#         .str.split(";")
#         .apply(lambda lst: [a.strip() for a in lst if a.strip()])
# )
# filtered_df["author_names"]

# filtered_df[
#    filtered_df["author_names"].apply(
#         lambda lst: any("Mandal" in a or "mandal" in a.lower() for a in lst)
#     )
# ]["author_names"]



In [ ]:
import re

def normalize_tokens(name):
    """Split a name into lowercase tokens (words)."""
    if not isinstance(name, str):
        return set()
    tokens = re.split(r"[,\s;\.\-]+", name.strip())
    return {t.lower() for t in tokens if t}
    

# Create token sets for each WVU faculty name
faculty_token_sets = [
    normalize_tokens(name)
    for name in wvu_faculty["Full Name"].dropna()
]

def author_matches_faculty(author):
    """Return True if this author matches ANY faculty name (≥2 tokens equal)."""
    tokens = normalize_tokens(author)
    if not tokens:
        return False
    
    for fac_tokens in faculty_token_sets:
        if len(tokens & fac_tokens) >= 2:
            return True
    return False
def row_has_wvu_faculty(authors):
    """Authors is a list, return True if any individual author matches faculty."""
    if not isinstance(authors, list):
        return False
    return any(author_matches_faculty(a) for a in authors)


In [ ]:
faculty_list = [
    (name, normalize_tokens(name))
    for name in wvu_faculty["Full Name"].dropna()
]
def matched_faculty_for_author(author):
    tokens = normalize_tokens(author)
    if not tokens:
        return []
    
    matches = []
    for fac_name, fac_tokens in faculty_list:
        if len(tokens & fac_tokens) >= 2:
            matches.append(fac_name)
    return matches
def row_matching_faculty(authors):
    if not isinstance(authors, list):
        return []
    
    matched = []
    for a in authors:
        matched.extend(matched_faculty_for_author(a))
    
    # Remove duplicates
    return list(set(matched))


In [ ]:
filtered_df["matched_faculty"] = filtered_df["author_names"].apply(row_matching_faculty)
filtered_df["has_wvu_faculty"] = filtered_df["matched_faculty"].apply(lambda x: len(x) > 0)
unique_faculty = set().union(*filtered_df["matched_faculty"])
print(len(unique_faculty))
sorted(unique_faculty)


In [ ]:
filtered_df["has_wvu_faculty"] = filtered_df["author_names"].apply(row_has_wvu_faculty)
matches_df = filtered_df[filtered_df["has_wvu_faculty"]]
matches_df["fund_acr"].isnull().sum()


In [ ]:
#mask = matches_df["fund_no"].astype(str).str.contains("1726534", na=False)
mask_dolly = matches_df.astype(str).apply(lambda col: col.str.contains("1726534", na=False)).any(axis=1)

len(matches_df[mask_dolly])


In [ ]:
# mask_dolly = matches_df["fund_no"].astype(str).str.contains("2117575", na=False)
mask_dolly = matches_df.astype(str).apply(lambda col: col.str.contains("2117575", na=False)).any(axis=1)
len(matches_df[mask_dolly])


In [ ]:
from agapi.client import Agapi

from openai import OpenAI

# model_name="deepseek-ai/deepseek-v3.1"
# model_name="google/gemma-3-27b-it"
# model_name="moonshotai/kimi-k2-instruct-0905"
# model_name="meta/llama-3.2-90b-vision-instruct"
# model_name="meta/llama-4-maverick-17b-128e-instruct"
# model_name = "openai/gpt-oss-20b"
# model_name="openai/gpt-oss-120b"
# model_name="qwen/qwen3-next-80b-a3b-instruct"

model_name = "openai/gpt-oss-20b"
api_key="sk-a70a04690a60449c8720138aca1c0eab"

client = OpenAI(
    base_url="https://atomgpt.org/api",
    api_key=api_key
)

result = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": '''Give just a topic to this abstract based on the ML tool used in two or one word : A framework for health monitoring of high-temperature boiler components has been developed using a hybrid grey box modeling technique that combines first-principles white box (WB) models with machine learning black box (BB) models. The WB model is a dynamic, distributed 3-D differential–algebraic equation model that includes detailed heat-transfer mechanisms across metal tube walls, while the BB model is a static-dynamic neural network model or a Bayesian machine learning model. Two hybrid grey box model structures are developed for modeling temporal and spatial variation in the main steam outlet temperature from the final superheater of a pulverized coal-fired boiler. An adaptive fading-memory-based training algorithm for the BB models is developed to address the build-up of model error with time while keeping the volume of data for training BB models constant. The framework is rigorously tested using data collected from a power plant under load-following operations. A key focus of this work is on the prediction of oxide scale growth, a significant damage mechanism in superheater tubing, using a parabolic growth law. Model predictions are compared against plant data for oxide scale measurements from tube specimens that were installed in a coal-fired boiler superheater for 22 months. The framework can be useful "
        for more efficient maintenance planning, cost reduction, and enhancing power plants reliability and safety.'''}
    ],
    reasoning_effort="high"
)

print(result.choices[0].message.content)


In [ ]:
import time

def get_topic(text):
    """Send a single abstract to the model and return the topic."""
    if not isinstance(text, str) or not text.strip():
        return ""

    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user",
                 "content": (
                     '''Give just a topic to this abstract based on the ML tool used in two or one word : '''
                     f"{text}"
                 )
                }
            ],
            reasoning_effort="high"
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print("Error:", e)
        return ""

# ---- Loop through description column safely ----
topics = []

for desc in matches_df["description"]:
    topic = get_topic(desc)
    topics.append(topic)
    time.sleep(0.3)  # small delay to avoid rate limits

matches_df["topic"] = topics

matches_df[["description", "topic"]].head(20)


In [ ]:
matches_df["generated_topic"].value_counts()

In [ ]:
matches_df["generated_topic"] = (
    matches_df["generated_topic"]
        .astype(str)
        .str.replace(r"[^A-Za-z\s]", "", regex=True)
        .str.strip()
)


In [ ]:
matches_df["generated_topic"].head(50)
topic_count = matches_df["generated_topic"].value_counts()
topic_count

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
topic_count[:10].plot(kind="bar")
plt.title("Normalized Topic Frequency")
plt.ylabel("Count")
plt.xlabel("Topic")
plt.tight_layout()
plt.show()

In [ ]:
TOPIC_NORMALIZE = {
    # Neural networks
    "neural network": ("Neural Network", "Neural Methods"),
    "NeuralNetworks": ("Neural Network", "Neural Methods"),
    "Neural networks": ("Neural Network", "Neural Methods"),
    "Networks": ("Neural Network", "Neural Methods"),
    "ann": ("Neural Network", "Neural Methods"),
    "dnn": ("Neural Network", "Neural Methods"),
    "deep learning": ("Neural Network", "Neural Methods"),
    "cnn": ("Neural Network", "Neural Methods"),
    "rnn": ("Neural Network", "Neural Methods"),
    "lstm": ("Neural Network", "Neural Methods"),
    "mlp": ("Neural Network", "Neural Methods"),
    

    # Bayesian
    "bayesian": ("Bayesian ML", "Bayesian Methods"),
    "bayesian ml": ("Bayesian ML", "Bayesian Methods"),
    "bayesian model": ("Bayesian ML", "Bayesian Methods"),
    "gaussian process": ("Bayesian ML", "Bayesian Methods"),
    "bayesian network": ("Bayesian ML", "Bayesian Methods"),

    # SVM
    "svm": ("SVM", "Kernel Methods"),
    "support vector machine": ("SVM", "Kernel Methods"),
    "support vector": ("SVM", "Kernel Methods"),

    # Tree & Ensemble
    "decision tree": ("Decision Tree", "Tree Ensemble"),
    "random forest": ("Random Forest", "Tree / Ensemble"),
    "xgboost": ("Boosting", "Tree / Ensemble"),
    "gradient boosting": ("Boosting", "Tree / Ensemble"),
    "gbm": ("Boosting", "Tree / Ensemble"),

    # Reinforcement Learning
    "reinforcement learning": ("Reinforcement Learning", "Reinforcement Learning"),
    "rl": ("Reinforcement Learning", "Reinforcement Learning"),

    # Clustering / Unsupervised
    "k-means": ("Clustering", "Unsupervised"),
    "kmeans": ("Clustering", "Unsupervised"),
    "clustering": ("Clustering", "Unsupervised"),

    # WTA
    "winner take all": ("WTA Network", "Neural Methods"),
    "winner-take-all": ("WTA Network", "Neural Methods"),
    "wta": ("WTA Network", "Neural Methods"),

    # Expert / Rule-Based
    "expert system": ("Expert System", "Rule-Based / Expert"),
    "rule-based": ("Expert System", "Rule-Based / Expert"),

    # Hybrid
    "grey box": ("Grey Box Model", "Hybrid / Grey-Box"),
    "gray box": ("Grey Box Model", "Hybrid / Grey-Box"),
    "hybrid model": ("Hybrid Model", "Hybrid / Grey-Box"),

    # Statistical / Regression
    "regression": ("Regression", "Statistical"),
    "statistical": ("Statistical Model", "Statistical"),
}



In [ ]:
def normalize_topic(raw_topic):
    if not isinstance(raw_topic, str):
        return (" ", "")

    t = raw_topic.lower().strip()

    # direct match
    if t in TOPIC_NORMALIZE:
        return TOPIC_NORMALIZE[t]

    # partial match
    for key in TOPIC_NORMALIZE:
        if key in t:
            return TOPIC_NORMALIZE[key]
    
    return (raw_topic.strip())



In [ ]:
matches_df[["topic_norm", "topic_category"]] = (
    matches_df["generated_topic"]
    .apply(normalize_topic)
    .apply(pd.Series)
)


In [ ]:
topic_counts = matches_df["topic_norm"].value_counts()
topic_counts[:10]


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
topic_counts[:10].plot(kind="bar")
plt.title("Normalized Topic Frequency")
plt.ylabel("Count")
plt.xlabel("Topic")
plt.tight_layout()
plt.show()


In [ ]:
funding_counts = matches_df["fund"].value_counts()

In [ ]:
funding_counts = matches_df["fund_sponsor"].value_counts()
plt.figure(figsize=(10,6))
funding_counts[:15].plot(kind="bar")
plt.title("Funding Source Frequency")
plt.ylabel("Count")
plt.xlabel("Funding Source")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

funding_counts = matches_df["fund_sponsor"].value_counts()
top15 = funding_counts[:15]

plt.figure(figsize=(12,6))

# Distinct colors for each bar
colors = plt.cm.tab20(np.linspace(0, 1, len(top15)))

bars = plt.bar(range(len(top15)), top15.values, color=colors)

plt.title("Funding Source Frequency")
plt.ylabel("Count")
plt.xlabel("Funding Source")

# Rotate long labels & shorten if too long
labels = [lbl if len(lbl) < 25 else lbl[:22] + "..." for lbl in top15.index]
plt.xticks(range(len(top15)), labels, rotation=45, ha="right")

# Add legend (key)
plt.legend(bars, top15.index, title="Funding Sources", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()
